In [8]:
from src.manage import DataLoader
import src.image as image
import src.heat_map as heat_map
import src.centers as centers
import pandas as pd

from matplotlib import pyplot as plt

In [10]:
DL = DataLoader()
DL.load_data("data")

In [11]:
def histogram(map, img):
    histogram = [0] * (map.max() + 1)

    for i in range(map.shape[0]):
        for j in range(map.shape[1]):
            if img[i][j] == 2:
                histogram[map[i][j]] += 1
    return histogram

In [12]:
def compute_features(heart):

    img = heart.get_img()
    cross_center = centers.cross_center(img)
    symetric_center = centers.symetric_center(img)
    #line_h = centers.longest_horizontal_line(img)
    line_v = centers.longest_vertical_line(img)

    cross_map = heat_map.create_map(img, cross_center)
    symetric_map = heat_map.create_map(img, symetric_center)
    #line_h_map = heat_map.create_line_map(img, line_h)
    line_v_map = heat_map.create_line_map(img, line_v)

    return {
        "id": heart.id,
        "label": heart.state,
        "corss_center_map": histogram(cross_map, img),
        "symetric_center_map": histogram(symetric_map, img),
        #"longest_horizontal_map": histogram(line_h_map, img),
        "longest_vertical_map": histogram(line_v_map, img)
    }

In [13]:
sample_heart_0 = DL.get_data()[7]
features_0 = compute_features(sample_heart_0)
print(sample_heart_0.is_cardiomegalia())

image.show(sample_heart_0.get_img())
plt.plot(features_0["corss_center_map"], label="cross center map")
plt.plot(features_0["symetric_center_map"], label="symetric center map")
plt.plot(features_0["longest_vertical_map"], label="longest vertical map")
plt.legend()
plt.title(f"heart {sample_heart_0.id} is not cardiomegalia")
plt.show()

sample_heart_1 = DL.get_data()[0]
print(sample_heart_1.is_cardiomegalia())
features_1 = compute_features(sample_heart_1)

image.show(sample_heart_1.get_img())
plt.plot(features_1["corss_center_map"], label="cross center map")
plt.plot(features_1["symetric_center_map"], label="symetric center map")
plt.plot(features_1["longest_vertical_map"], label="longest vertical map")
plt.legend()
plt.title(f"heart {sample_heart_1.id} is cardiomegalia")
plt.show()

IndexError: list index out of range

In [14]:
def pixels_distribution(heart): #btw this only maps histogram values to 5
    img = heart.get_img()
    line_v = centers.longest_vertical_line(img)
    line_v_map = heat_map.create_line_map(img, line_v)

    his = histogram(line_v_map, img)
    all = sum(his)
    n = len(his)
    part_size = n // 5

    x1 = sum(his[:part_size])/all
    x2 = sum(his[part_size : 2 * part_size])/all
    x3 = sum(his[2 * part_size : 3 * part_size])/all
    x4 = sum(his[3 * part_size : 4 * part_size])/all
    x5 = sum(his[4 * part_size :])/all

    return {"id": heart.id, "label": heart.is_cardiomegalia(), "x1": x1, "x2": x2, "x3": x3, "x4": x4, "x5": x5}

In [15]:
# Test feature of counting pixels in
feat = []
FILE_NAME = "pixel_dist_5.csv"
if len(DL.get_data()) > 0:
    for hear in DL.get_data():
        print(f"processing heart {hear.id}")
        feat.append(pixels_distribution(hear))
        print(pixels_distribution(hear))

    df = pd.DataFrame(feat)
    df.to_csv(f"results\\{FILE_NAME}", index=False)


else:
    print("No data")

No data
